<a href="https://colab.research.google.com/github/gabrielsanchez/sussex-thesis/blob/main/Neutral_atom_noise_model_and_syndrome_dataset_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neutral-atom noise model and syndrome dataset generation

**Thesis context:** generating training data for a neural decoder
specialized to neutral-atom surface codes.



## 1. Noise model

Parameters are intended to be in a realistic regime for current
neutral-atom hardware: two-qubit gate fidelities
around 99.5%, atom loss per gate around $10^{-3}$ to $10^{-2}$.

Real hardware noise is complex. For simulation, a **Pauli-twirled** noise model is used by averaging over the random phases introduced by noise, reducing any noise channel to a probabilistic mixture of Pauli errors (I, X, Y, Z) [1]. This is an approximation that is widely used because it lets us use fast stabilizer simulators like **Stim** [2].

### 1.1 Pauli errors

The four Pauli matrices are:
- **I** (Identity): nothing happens
- **X** (bit-flip): |0⟩ ↔ |1⟩, like a classical bit flip
- **Y** (combined): both X and Z simultaneously
- **Z** (phase-flip): |+⟩ ↔ |−⟩, invisible in the computational basis but detectable with the right ancilla

An X error on a data qubit flips that qubit's value. A Z error flips its phase. Both are detectable via the stabilizer measurements, but each leaves a different signature in the syndrome.

### 1.2 Depolarizing noise

Single-qubit depolarizing (DEPOLARIZE1 at rate p): after a single-qubit gate, with probability p the qubit is hit by a uniformly random Pauli error (X, Y, or Z, each with probability p/3) [1]. This channel is commonly used as an effective model for stochastic control imperfections such as laser intensity fluctuations, spontaneous-emission-induced noise, and thermal fluctuations [3].

**Two-qubit depolarizing** (`DEPOLARIZE2` at rate p): after a two-qubit gate, a random two-qubit Pauli (one of 15 non-identity options) is applied [1]. Models crosstalk, imperfect gate pulses, etc.

**Measurement noise** (`X_ERROR` at rate p before measurement): the measurement outcome is flipped with probability p [1]. Models photon-counting errors in fluorescence readout.

**Reset noise** (`X_ERROR` at rate p after reset): the reset operation fails to fully reinitialize the qubit with probability p.

### 1.2 Depolarizing noise

Single-qubit depolarizing (DEPOLARIZE1 at rate p): after a single-qubit gate, with probability p the qubit is hit by a uniformly random Pauli error (X, Y, or Z, each with probability p/3) [1]. This channel is commonly used as an effective model for stochastic control imperfections such as laser intensity fluctuations, spontaneous-emission-induced noise, and thermal fluctuations [3].

**Two-qubit depolarizing** (`DEPOLARIZE2` at rate p): after a two-qubit gate, a random two-qubit Pauli (one of 15 non-identity options) is applied [1]. Models crosstalk, imperfect gate pulses, etc.

**Measurement noise** (`X_ERROR` at rate p before measurement): the measurement outcome is flipped with probability p [1]. Models photon-counting errors in fluorescence readout.

**Reset noise** (`X_ERROR` at rate p after reset): the reset operation fails to fully reinitialize the qubit with probability p.

### 1.3 Correlated ZZ errors and the Rydberg drive

This is the most neutral-atom-specific noise channel in the model.

When two atoms undergo a Rydberg gate, they are both coupled to the same global laser beam. If the laser power fluctuates slightly, *both* atoms are affected by a common-mode phase error simultaneously. This produces a **correlated ZZ error**: both qubits simultaneously experience a Z error.

In Stim, this is modelled with `CORRELATED_ERROR` targeting `Z ⊗ Z` on both qubits of each two-qubit gate, with probability `p_correlated_zz` [1].

Correlated errors are more dangerous than independent errors because they can create error patterns that span multiple stabilizers and fool the decoder. A single-qubit error might be easy to locate; a correlated two-qubit error can look like two separate single-qubit errors in two different places.

At the default rate (`1e-3`), this channel is at the noise floor, meaning it barely affects results. But increasing it (to model a poorly calibrated laser) can significantly degrade the code's performance.

### 1.4 Erasure (atom loss) as a side-channel

When an atom is lost during a two-qubit gate, its qubit effectively becomes a completely random state. This is equivalent to a **maximally depolarizing channel** of strength 0.75 (a standard result from quantum information theory: a maximally mixed qubit has 3/4 probability of any given Pauli error being applied).

However, atom loss is also *heralded*: the experiment detects which atom was lost. This herald is a separate classical signal, distinct from the syndrome measurements.

### Sources

TODO: Format correctly

[1] https://github.com/quantumlib/Stim/wiki/Stim-v1.10-Gate-Reference

[2] Stim: a fast stabilizer circuit simulator

[3] https://www.nature.com/articles/s41534-022-00586-4

[4] Cong et al., *Hardware-efficient, fault-tolerant quantum computation with Rydberg atoms*, PRX Quantum 3, 030339 (2022)

[5] Sahay et al., *Tailoring quantum error correction to spin-qubit hardware*, PRX Quantum 4, 040334 (2023)

[6] Evered et al., *High-fidelity parallel entangling gates on a neutral-atom quantum computer*, Nature 622, 268–272 (2023)

[7] Graham et al., *Multi-qubit entanglement and algorithms on a neutral-atom quantum computer*, Nature 604, 457–462 (2022)

[8] Bluvstein et al., *Logical quantum processor based on reconfigurable atom arrays*, Nature 626, 58–65 (2024)

In [ ]:
!pip install stim pymatching

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.2/626.2 kB 19.5 MB/s eta 0:00:00


In [ ]:
import json
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import stim
import pymatching
import matplotlib.pyplot as plt

print("stim", stim.__version__)
print("pymatching", pymatching.__version__)
print("numpy", np.__version__)

stim 1.16.0
pymatching 2.4.0
numpy 2.0.2


In [ ]:
@dataclass
class NeutralAtomNoise:
    """
    Noise parameters for the neutral-atom surface code.

    Two categories of noise are modeled:

    1. Generic Pauli-twirled noise (common to most qubit platforms):
       depolarizing errors, measurement bitflips, and reset errors.

    2. Neutral-atom-specific noise:
       - Correlated ZZ errors from shared Rydberg laser drives.
       - Atom loss (erasure), where a qubit physically falls out of its trap.
    """

    # -------------------------------------------------------------------------
    # Standard (platform-agnostic) noise rates
    # -------------------------------------------------------------------------

    # Probability that a single-qubit gate introduces a random Pauli error.
    # Modeled as a depolarizing channel: the qubit is pushed into a random
    # Pauli eigenstate with this probability.
    p_depol1: float = 1e-3          # single-qubit depolarizing

    # Same as above but for two-qubit gates (e.g. CX, CZ).
    # Typically higher than single-qubit because entangling gates are slower
    # and harder to calibrate.
    p_depol2: float = 5e-3          # two-qubit depolarizing

    # Probability that a measurement outcome is flipped (bitflip).
    # This captures readout errors: the qubit was in |0⟩ but we record "1",
    # or vice versa.
    p_meas: float = 5e-3            # measurement bitflip

    # Probability that a reset operation leaves the qubit in the wrong state.
    # After an MR (measure-and-reset) instruction, the qubit should be |0⟩;
    # this is the probability it ends up in |1⟩ instead.
    p_reset: float = 1e-3           # reset bitflip

    # -------------------------------------------------------------------------
    # Neutral-atom-specific noise rates
    # -------------------------------------------------------------------------

    # When multiple qubits share a Rydberg laser drive, they can accumulate
    # correlated phase (Z) errors together. This is a two-body ZZ error:
    # both qubits flip their Z component simultaneously.
    p_correlated_zz: float = 1e-3   # correlated ZZ on shared Rydberg drive

    # In neutral-atom hardware, atoms are held in optical traps.
    # Occasionally an atom is lost from its trap entirely during a two-qubit
    # gate. This "erasure" error is detectable in principle (the trap is now
    # empty), which is why it is tracked separately from generic depolarizing.
    p_erasure: float = 2e-3         # per-qubit atom loss per 2q gate

    # When an atom is lost, it is replaced with a fresh (uninitialized) atom,
    # which is effectively in a maximally mixed state.  This is approximated by
    # applying a depolarizing channel of strength:
    #     erasure_depol_strength * p_erasure
    # to that qubit. The factor < 1 captures that not every axis is equally
    # affected when the atom re-thermalizes.
    erasure_depol_strength: float = 0.75

## TODO: Inject the errors, generate dataset, and do sanity check against pymatching